# Alytes-ReID — Setup & Training

**Who runs this notebook**: researcher / developer (once per project, or when new labeled data arrives).

**What it does**:
1. Install dependencies
2. Download toad photos from iNaturalist (no account needed)
3. Auto-annotate images using YOLO-World (open-vocabulary detector)
4. Fine-tune YOLO11 on toad-specific data
5. Validate SAM2 segmentation
6. Train Re-ID model (when labeled data is available)
7. Save models to Google Drive → used by `02_toad_reid.ipynb`

**No manual annotation required** — YOLO-World finds toads by text description ("toad", "frog").

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/danort92/Alytes-ReID/blob/claude/alytes-reid-system-qgved/notebooks/01_setup_and_training.ipynb)

---
## 1. Environment Setup

Run this cell once. It installs everything needed and connects Google Drive.

In [ ]:
# Check GPU
!nvidia-smi

# Clone repo
!git clone -b claude/alytes-reid-system-qgved https://github.com/danort92/Alytes-ReID.git 2>/dev/null || (cd Alytes-ReID && git pull)
%cd Alytes-ReID

# Install dependencies
!pip install -r requirements.txt -q

# Mount Google Drive (to save trained models)
from google.colab import drive
drive.mount('/content/drive')

DRIVE_MODEL_DIR = '/content/drive/MyDrive/Alytes-ReID/models'
!mkdir -p {DRIVE_MODEL_DIR}

print('\nSetup complete!')

---
## 2. Download Toad Photos from iNaturalist

Downloads research-grade photos of *Alytes obstetricans* from iNaturalist.  
No account or API key needed — the iNaturalist API is public.

In [ ]:
from pathlib import Path
from src.data.download_inat import download_alytes_images

MAX_IMAGES = 500  # increase to 1000+ for better results (takes longer)

inat_images = download_alytes_images(
    output_dir=Path('data/raw/inaturalist'),
    max_images=MAX_IMAGES,
)
print(f'\nDownloaded {len(inat_images)} images')

---
## 3. Preview Downloaded Images

In [ ]:
import random
import cv2
import matplotlib.pyplot as plt

sample = random.sample(inat_images, min(8, len(inat_images)))

fig, axes = plt.subplots(2, 4, figsize=(16, 8))
for ax, img_path in zip(axes.flat, sample):
    img = cv2.cvtColor(cv2.imread(str(img_path)), cv2.COLOR_BGR2RGB)
    ax.imshow(img)
    ax.set_title(img_path.name[:20], fontsize=8)
    ax.axis('off')
plt.suptitle('Sample iNaturalist Images (Alytes obstetricans)')
plt.tight_layout()
plt.show()

print(f'Total images available: {len(inat_images)}')

---
## 4. Auto-Annotate Images

Uses **YOLO-World** (open-vocabulary detector) to find toads/frogs in photos
by text description — much more robust than fixed COCO classes, which miss
camouflaged toads in field photos.

No manual annotation needed! Detections are saved as YOLO-format label files.

In [ ]:
# Ensure ultralytics is installed (may not be present in Colab by default)
!pip install ultralytics -q

from src.data.auto_annotate import auto_annotate

stats = auto_annotate(
    images_dir=Path('data/raw/inaturalist'),
    output_dir=Path('data/raw/auto_labels'),
    model_name='yolov8s-worldv2',  # open-vocabulary detector
    confidence=0.10,               # balanced: good recall, few false positives
    text_classes=['toad', 'frog', 'amphibian'],
)

print(f'\n=== Auto-Annotation Results ===')
print(f"  Images processed:  {stats['total_images']}")
print(f"  Images annotated:  {stats['annotated']}")
print(f"  Images skipped:    {stats['skipped']}  (no toad/frog detected)")
print(f"  Total bounding boxes: {stats['total_boxes']}")

### Preview auto-annotations

Visual check: do the bounding boxes look correct?

In [ ]:
from src.utils.visualization import draw_yolo_labels

# Find images that got annotated
label_dir = Path('data/raw/auto_labels')
annotated_images = [
    img for img in inat_images
    if (label_dir / f'{img.stem}.txt').exists()
]

preview = random.sample(annotated_images, min(6, len(annotated_images)))

fig, axes = plt.subplots(2, 3, figsize=(15, 10))
for ax, img_path in zip(axes.flat, preview):
    img = cv2.cvtColor(cv2.imread(str(img_path)), cv2.COLOR_BGR2RGB)
    label_path = label_dir / f'{img_path.stem}.txt'
    img_with_boxes = draw_yolo_labels(img, label_path)
    ax.imshow(img_with_boxes)
    ax.set_title(img_path.name[:25], fontsize=8)
    ax.axis('off')
plt.suptitle('Auto-annotated images (green boxes = detected toads)')
plt.tight_layout()
plt.show()

---
## 5. Prepare Dataset for Training

Splits auto-annotated images into train/val/test sets in YOLO format.

In [ ]:
from src.data.prepare_dataset import split_dataset, create_yolo_dataset_yaml

counts = split_dataset(
    images_dir=Path('data/raw/inaturalist'),
    labels_dir=Path('data/raw/auto_labels'),
    output_dir=Path('data/processed/detection'),
)

dataset_yaml = create_yolo_dataset_yaml(
    dataset_dir=Path('data/processed/detection'),
    classes=['toad'],
)

print(f'\nDataset splits: {counts}')
print(f'Dataset config: {dataset_yaml}')

---
## 6. Fine-Tune YOLO11 for Toad Detection

Fine-tunes YOLO11 on the auto-annotated toad dataset.  
Starting from COCO weights gives the model a strong head start.

In [ ]:
# Training settings — adjust if needed
YOLO_MODEL = 'yolo11s'  # yolo11n (fastest) | yolo11s | yolo11m | yolo11l | yolo11x
EPOCHS = 100
BATCH_SIZE = 16  # reduce to 8 if you get GPU out-of-memory errors

print(f'Model: {YOLO_MODEL}, Epochs: {EPOCHS}, Batch size: {BATCH_SIZE}')

In [ ]:
from src.detection.train import load_config, train_detector

config = load_config(Path('config/detection.yaml'))

# Override config with the values set above
config['model']['architecture'] = YOLO_MODEL
config['training']['epochs'] = EPOCHS
config['training']['batch_size'] = BATCH_SIZE

# Train
best_weights = train_detector(config)
print(f'\nBest weights saved to: {best_weights}')

---
## 7. Evaluate Detection

In [ ]:
from src.detection.evaluate import evaluate_model

metrics = evaluate_model(best_weights, config)

print('\n=== Detection Results ===')
for name, value in metrics.items():
    print(f'  {name:15s}: {value:.4f}')

---
## 8. SAM2 Segmentation Validation

In [ ]:
# Install SAM2 from GitHub (not available on PyPI)
!pip install git+https://github.com/facebookresearch/sam2.git -q

In [ ]:
from src.segmentation.segment import ToadSegmenter
from src.detection.predict import load_detector, detect_toads, get_best_detection
from src.utils.visualization import draw_detections, draw_mask_overlay

detector = load_detector(best_weights)
segmenter = ToadSegmenter()  # loads SAM2 lazily on first call

# Pick a test image
test_path = inat_images[0]
test_img = cv2.cvtColor(cv2.imread(str(test_path)), cv2.COLOR_BGR2RGB)

detections = detect_toads(detector, test_path)
best_det = get_best_detection(detections)

if best_det:
    cropped, mask = segmenter.segment_and_crop(test_img, best_det['bbox'])
    overlay = draw_mask_overlay(
        draw_detections(test_img, detections),
        segmenter.segment_from_bbox(test_img, best_det['bbox'])
    )
    fig, axes = plt.subplots(1, 3, figsize=(15, 5))
    axes[0].imshow(test_img); axes[0].set_title('Original')
    axes[1].imshow(overlay); axes[1].set_title('Detection + Mask')
    axes[2].imshow(cropped); axes[2].set_title('Cropped')
    for ax in axes: ax.axis('off')
    plt.tight_layout()
    plt.show()
else:
    print('No toad detected in test image.')

---
## 8b. Test Pipeline on Biologist's Photos

**Important**: The iNaturalist images above are field photos (natural backgrounds, various angles).
The biologist's actual data looks very different: **dorsal view, flash, uniform cardboard background**.

Upload a few sample photos from the biologist to verify the full pipeline
(detection → segmentation → preprocessing) works correctly on the real data format.

In [ ]:
from google.colab import files as colab_files
from src.preprocessing.pipeline import load_config as load_preproc_config, preprocess_toad

# Upload biologist's sample photos (dorsal view, cardboard background)
print("Upload 1-3 sample photos from the biologist:")
uploaded = colab_files.upload()

preproc_config = load_preproc_config(Path('config/preprocessing.yaml'))

n = len(uploaded)
fig, axes = plt.subplots(n, 4, figsize=(20, 5 * n))
if n == 1:
    axes = axes[None, :]  # ensure 2D

for row, filename in enumerate(uploaded):
    img_path = Path(filename)
    img = cv2.cvtColor(cv2.imread(str(img_path)), cv2.COLOR_BGR2RGB)

    # Step 1: Detection
    detections = detect_toads(detector, img_path)
    best_det = get_best_detection(detections)

    axes[row, 0].imshow(img)
    axes[row, 0].set_title(f'Original: {filename[:25]}', fontsize=9)

    if best_det:
        # Step 2: Segmentation
        mask = segmenter.segment_from_bbox(img, best_det['bbox'])
        cropped, cropped_mask = segmenter.segment_and_crop(img, best_det['bbox'])

        # Step 3: Full preprocessing
        standardized = preprocess_toad(cropped, cropped_mask, preproc_config)

        overlay = draw_mask_overlay(draw_detections(img, detections), mask)
        axes[row, 1].imshow(overlay)
        axes[row, 1].set_title(f'Detection + Mask (conf={best_det["confidence"]:.2f})', fontsize=9)

        axes[row, 2].imshow(cropped)
        axes[row, 2].set_title('Cropped (background removed)', fontsize=9)

        axes[row, 3].imshow(standardized)
        axes[row, 3].set_title('Standardized (256x256, ready for Re-ID)', fontsize=9)
    else:
        for c in range(1, 4):
            axes[row, c].text(0.5, 0.5, 'No toad detected', ha='center', va='center', fontsize=12)

    for ax in axes[row]:
        ax.axis('off')

plt.suptitle("Biologist's Photos — Full Pipeline Test", fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print('\nCheck that:')
print('  1. Detection finds the toad (green box covers it)')
print('  2. Segmentation mask follows the toad outline (not the cardboard)')
print('  3. Standardized image shows only the toad, centered, on black background')

---
## 8c. Import Biologist's Full Dataset

Upload the biologist's ZIP archive containing the field photos.
The import script automatically:
- Extracts all images from the ZIP
- Removes duplicates when both JPG and CR3 (Canon RAW) exist for the same photo
- Keeps the JPG version (smaller, ready to use)
- Reports statistics on the imported dataset

In [ ]:
from google.colab import files as colab_files
from src.data.import_biologist_data import import_from_zip

print("Upload the biologist's ZIP archive (containing JPG and/or CR3 files):")
uploaded = colab_files.upload()

zip_name = list(uploaded.keys())[0]
print(f'\nProcessing: {zip_name}')

bio_stats = import_from_zip(
    zip_path=Path(zip_name),
    output_dir=Path('data/raw/biologist'),
)

print(f'\n=== Import Results ===')
print(f"  Images in archive:   {bio_stats['total_in_archive']}")
print(f"  Duplicates removed:  {bio_stats['duplicates_removed']}")
print(f"  Images extracted:    {bio_stats['images_extracted']}")
print(f"    JPG:  {bio_stats['jpg_count']}")
print(f"    RAW:  {bio_stats['raw_count']}")

if bio_stats['duplicates_removed'] > 0:
    print(f"\n  Skipped (duplicates): {bio_stats['skipped_files'][:5]}")
    if bio_stats['duplicates_removed'] > 5:
        print(f"    ... and {bio_stats['duplicates_removed'] - 5} more")

In [ ]:
# Preview imported images
bio_images = sorted(Path('data/raw/biologist').glob('*'))
preview = random.sample(bio_images, min(8, len(bio_images)))

fig, axes = plt.subplots(2, 4, figsize=(16, 8))
for ax, img_path in zip(axes.flat, preview):
    img = cv2.cvtColor(cv2.imread(str(img_path)), cv2.COLOR_BGR2RGB)
    ax.imshow(img)
    ax.set_title(img_path.name[:20], fontsize=8)
    ax.axis('off')
plt.suptitle(f"Biologist's Photos ({len(bio_images)} images)", fontsize=14)
plt.tight_layout()
plt.show()

### Auto-annotate biologist's photos

Run YOLO-World on the biologist's photos to generate bounding box labels.
These will be merged with the iNaturalist labels for retraining.

In [ ]:
bio_stats_annot = auto_annotate(
    images_dir=Path('data/raw/biologist'),
    output_dir=Path('data/raw/biologist_labels'),
    model_name='yolov8s-worldv2',
    confidence=0.10,
    text_classes=['toad', 'frog', 'amphibian'],
)

print(f'\n=== Biologist Photos — Auto-Annotation ===')
print(f"  Images processed:  {bio_stats_annot['total_images']}")
print(f"  Images annotated:  {bio_stats_annot['annotated']}")
print(f"  Images skipped:    {bio_stats_annot['skipped']}")
print(f"  Total boxes:       {bio_stats_annot['total_boxes']}")

# Preview annotations
bio_label_dir = Path('data/raw/biologist_labels')
bio_annotated = [
    img for img in bio_images
    if (bio_label_dir / f'{img.stem}.txt').exists()
]

if bio_annotated:
    preview_ann = random.sample(bio_annotated, min(6, len(bio_annotated)))
    fig, axes = plt.subplots(2, 3, figsize=(15, 10))
    for ax, img_path in zip(axes.flat, preview_ann):
        img = cv2.cvtColor(cv2.imread(str(img_path)), cv2.COLOR_BGR2RGB)
        label_path = bio_label_dir / f'{img_path.stem}.txt'
        img_with_boxes = draw_yolo_labels(img, label_path)
        ax.imshow(img_with_boxes)
        ax.set_title(img_path.name[:25], fontsize=8)
        ax.axis('off')
    plt.suptitle("Biologist's Photos — Auto-annotations")
    plt.tight_layout()
    plt.show()
else:
    print('\nNo toads detected. Manual annotation may be needed for these images.')

---
## 8d. Retrain YOLO with Combined Dataset

Merges iNaturalist + biologist's photos into a single dataset and retrains YOLO.
This addresses the **domain gap**: the original model was trained only on field photos
and may struggle with the biologist's standardized cardboard-background images.

In [ ]:
import shutil as _shutil

# Merge biologist images + labels into the existing dataset
merged_images = Path('data/raw/merged_images')
merged_labels = Path('data/raw/merged_labels')
merged_images.mkdir(parents=True, exist_ok=True)
merged_labels.mkdir(parents=True, exist_ok=True)

# Copy iNaturalist images + labels
inat_dir = Path('data/raw/inaturalist')
inat_lbl = Path('data/raw/auto_labels')
for img in inat_dir.glob('*'):
    if img.suffix.lower() in {'.jpg', '.jpeg', '.png'}:
        lbl = inat_lbl / f'{img.stem}.txt'
        if lbl.exists():
            _shutil.copy2(img, merged_images / img.name)
            _shutil.copy2(lbl, merged_labels / f'{img.stem}.txt')

# Copy biologist images + labels
bio_dir = Path('data/raw/biologist')
bio_lbl = Path('data/raw/biologist_labels')
for img in bio_dir.glob('*'):
    if img.suffix.lower() in {'.jpg', '.jpeg', '.png'}:
        lbl = bio_lbl / f'{img.stem}.txt'
        if lbl.exists():
            _shutil.copy2(img, merged_images / img.name)
            _shutil.copy2(lbl, merged_labels / f'{img.stem}.txt')

n_inat = len(list(inat_lbl.glob('*.txt')))
n_bio = len(list(bio_lbl.glob('*.txt')))
n_merged = len(list(merged_labels.glob('*.txt')))
print(f'Merged dataset: {n_inat} iNaturalist + {n_bio} biologist = {n_merged} total')

# Split merged dataset
counts = split_dataset(
    images_dir=merged_images,
    labels_dir=merged_labels,
    output_dir=Path('data/processed/detection_v2'),
)

dataset_yaml_v2 = create_yolo_dataset_yaml(
    dataset_dir=Path('data/processed/detection_v2'),
    classes=['toad'],
)

print(f'New dataset splits: {counts}')
print(f'Dataset config: {dataset_yaml_v2}')

In [ ]:
# Retrain YOLO on merged dataset
config_v2 = load_config(Path('config/detection.yaml'))
config_v2['model']['architecture'] = YOLO_MODEL
config_v2['training']['epochs'] = EPOCHS
config_v2['training']['batch_size'] = BATCH_SIZE
config_v2['data']['dataset_dir'] = 'data/processed/detection_v2'

best_weights_v2 = train_detector(config_v2)
print(f'\nRetrained model saved to: {best_weights_v2}')

# Use the retrained model for subsequent steps
best_weights = best_weights_v2
print('Updated best_weights to use the retrained model.')

---
## 9. Re-ID Model Training

Requires labeled data organized as `data/processed/reid/<INDIVIDUAL_ID>/<image>.png`.  
Upload the biologist's photos and run the preparation script, then train.

In [ ]:
# Upload labeled re-ID data from local machine
# from google.colab import files
# uploaded = files.upload()  # upload a zip archive
# !unzip -q your_labeled_data.zip -d data/processed/reid/

# Or copy from Drive:
# !cp -r '/content/drive/MyDrive/Alytes-ReID/reid_data' data/processed/reid/

In [ ]:
# Train Re-ID model
# from src.reid.train import train_reid
#
# reid_config = load_config(Path('config/reid.yaml'))
# reid_model_path = train_reid(reid_config, data_dir=Path('data/processed/reid'))
# print(f'Re-ID model saved to: {reid_model_path}')

---
## 10. Save Models to Google Drive

In [ ]:
import shutil

# Detection model
shutil.copy2(str(best_weights), f'{DRIVE_MODEL_DIR}/detection_best.pt')
print(f'Detection model saved to Drive: {DRIVE_MODEL_DIR}/detection_best.pt')

# Re-ID model (uncomment after training)
# shutil.copy2(str(reid_model_path), f'{DRIVE_MODEL_DIR}/reid_model.pt')

# Re-ID database (uncomment after building)
# shutil.copytree('data/models/reid', f'{DRIVE_MODEL_DIR}/reid_db', dirs_exist_ok=True)

print('Models saved to Google Drive. Ready to use in 02_toad_reid.ipynb')